# 04 · ServiceBusProcessor

`ServiceBusProcessor` is the **event-driven** receive API. You wire up:

- `ProcessMessageAsync` — your message handler (called concurrently per `MaxConcurrentCalls`)
- `ProcessErrorAsync` — your error handler

The processor handles lock renewal, retries, and concurrency for you. This is what you'd use in a hosted service / worker in production.


In [ ]:
#r "nuget: Azure.Messaging.ServiceBus, 7.18.2"

In [ ]:
#!import ../src/shared/Config.cs

In [ ]:
using Azure.Messaging.ServiceBus;

var client = new ServiceBusClient(Config.ConnectionString);
var sender = client.CreateSender(Config.QueueName);

// seed some work
await sender.SendMessagesAsync(Enumerable.Range(1, 20)
    .Select(i => new ServiceBusMessage($"work-item-{i}")));
Console.WriteLine("Seeded 20 work items.");

## 1. Configure the processor

In [ ]:
var processor = client.CreateProcessor(Config.QueueName, new ServiceBusProcessorOptions
{
    MaxConcurrentCalls = 4,                     // parallel handlers
    AutoCompleteMessages = true,                // Complete on successful return
    MaxAutoLockRenewalDuration = TimeSpan.FromMinutes(5)
});

int processed = 0;

processor.ProcessMessageAsync += async args =>
{
    Interlocked.Increment(ref processed);
    Console.WriteLine($"[{Environment.CurrentManagedThreadId}] {args.Message.Body}");
    await Task.Delay(100); // simulate work
};

processor.ProcessErrorAsync += args =>
{
    Console.WriteLine($"ERROR from {args.EntitySource}: {args.Exception.Message}");
    return Task.CompletedTask;
};

## 2. Start, let it drain, stop

In [ ]:
await processor.StartProcessingAsync();
await Task.Delay(TimeSpan.FromSeconds(5));
await processor.StopProcessingAsync();

Console.WriteLine($"\nProcessed {processed} messages.");

await processor.DisposeAsync();
await sender.DisposeAsync();
await client.DisposeAsync();

## Production tips

- The processor **automatically renews locks** up to `MaxAutoLockRenewalDuration`. Set this longer than your slowest expected handler.
- Set `AutoCompleteMessages = false` if you want to call `CompleteMessageAsync` / `DeadLetterMessageAsync` explicitly inside the handler (more control).
- `MaxConcurrentCalls` is per-receiver. To scale further, run multiple processor instances on multiple hosts.

Next: [`05-dead-letter.ipynb`](05-dead-letter.ipynb)
